<a href="https://colab.research.google.com/github/Andrea-24744/Procesos-Est-casticos-/blob/main/M%C3%A9todo_de_Uniformizaci%C3%B3n_para_una_CMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Método de Uniformización para una CMTC




> Andrea Santelices Medina

---









###Ejercicio 3
**Teorema $($matriz $P(t))$**: La matriz de probabilidad de transición $P(t) = [pi,j (t)]$ está dada por
$$
P(t)=\sum_{k=0}^{\infty} e^{-rt}\,\frac{(rt)^k}{k!}\,P^k
$$

$$
M \approx \max\{rt+5\sqrt{rt},\,20\}
$$

1. Use esta propuesta para calcular $P(0.5), P(1)$ y $P(5)$ para la matriz $R$ del ejercicio 1
2. ¿Se verifica la ecuación de Chapman-Kolmogorov $P(1) = P(0.5)P(0.5)$

In [7]:
import numpy as np
from math import exp, factorial, sqrt
import sympy as sp

# MATRIZ DE TASAS
R = sp.Matrix([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
])

# MATRIZ UNIFORMIZADA
def matriz_uniformizada(R):
    n = R.rows
    # tasas de salida
    ri = []
    for i in range(n):
        ri.append(sum(R.row(i)))
    # r = max_i ri
    r = max(ri)
    P_hat = sp.zeros(n)
    for i in range(n):
        for j in range(n):
            if i == j:
                P_hat[i, j] = 1 - sp.Rational(ri[i], r)
            else:
                P_hat[i, j] = sp.Rational(R[i, j], r)
    return P_hat, r


# APROXIMACIÓN DE P(t)
def P_t(t, P_hat, r):
    M = int(max(r*t + 5*sp.sqrt(r*t), 20))
    n = P_hat.rows
    P = sp.zeros(n)
    for k in range(M + 1):
        coef = sp.exp(-r*t)*(r*t)**k/sp.factorial(k)
        P += coef*(P_hat**k)
    return sp.N(P, 8)


# PROGRAMA PRINCIPAL
P_hat, r = matriz_uniformizada(R)
print("P_hat =")
sp.pprint(P_hat)
print("\nr =", r)


# P(0.5)
P05 = P_t(sp.Rational(1, 2), P_hat, r)
print("\nP(0.5)=")
sp.pprint(P05)



# P(1)
P1 = P_t(1, P_hat, r)
print("\nP(1)=")
sp.pprint(P1)


# P(5)
P5 = P_t(5, P_hat, r)
print("\nP(5)=")
sp.pprint(P5)


# CHAPMAN-KOLMOGOROV
CK = P05*P05
print("\nP(0.5)^2 =")
sp.pprint(sp.N(CK, 8))

print("\nError máximo:")

error = max(abs((P1 - CK)[i])
            for i in range(P1.rows*P1.cols))

print(sp.N(error))

P_hat =
⎡1/6  1/3  1/2   0 ⎤
⎢                  ⎥
⎢2/3   0   1/3   0 ⎥
⎢                  ⎥
⎢ 0   1/3  1/3  1/3⎥
⎢                  ⎥
⎣1/6   0   1/2  1/3⎦

r = 6

P(0.5)=
⎡0.25060868  0.2169646   0.38665694  0.14576979⎤
⎢                                              ⎥
⎢0.25313484  0.23836098  0.37440924  0.13409493⎥
⎢                                              ⎥
⎢0.1691195   0.19361489  0.42030102  0.2169646 ⎥
⎢                                              ⎥
⎣0.15801748  0.15744464  0.39833179  0.28620609⎦

P(1)=
⎡0.20615112  0.20390203  0.3987096   0.1912358 ⎤
⎢                                              ⎥
⎢0.20828421  0.2053407   0.39789917  0.18847446⎥
⎢                                              ⎥
⎢0.19675849  0.19837934  0.40095869  0.20390203⎥
⎢                                              ⎥
⎣0.19204622  0.19399715  0.40147094  0.21248423⎦

P(5)=
⎡0.19999925  0.19999925  0.3999985  0.19999925⎤
⎢                                             ⎥
⎢0.19999925  0.19999925  0.399998

###Ejercicio 4

**Teorema (Cotas de error)** para $P(t)$: Para un $t ≥ 0$ fijo, sea:

$$
P^M(t)=\left[p_{ij}^{M}(t)\right]
=\sum_{k=0}^{M} e^{-rt}\,\frac{(rt)^k}{k!}\,P^k
$$

entonces
$$
\left|p_{ij}(t)-p_{ij}^{M}(t)\right|
\leq
\sum_{k=M+1}^{\infty}
e^{-rt}\,\frac{(rt)^k}{k!}
$$

Este teorema se puede usar así:

Suponga que se desea calcular
$P(t)$
con una tolerancia ε. Elija $M$ tal que
$$\sum_{k=M+1}^{\infty}
e^{-rt}\,\frac{(rt)^k}{k!} \leq ϵ$$

Y se puede implementar de acuerdo al siguiente algoritmo de uniformización para $P(t):$

1. Dados $R, t, 0 < ε < 1$
2. Calcular $r$ usando la igualdad en la definición.
3. Calcular $\hat{P}$.
4. $A = \hat{P}; B = e^{−rt}I; c = e^{−rt}; sum = c; k = 1$
5. Mientras $sum < 1 − ε$ hacer:
    *   $A = A\hat{P}$
    *   $c = c ∗ (rt)/k$
    *   $B = B + cA$
    *   $sum = sum + c$
    *   $k = k + 1$
6. $B$ está a ε de $P(t).$

In [3]:
import sympy as sp

def uniformizacion(R, t, eps=1e-5):
    n = R.rows

    # r
    ri = [sum(R.row(i)) for i in range(n)]
    r = max(ri)


    # P_hat
    P_hat = sp.zeros(n)
    for i in range(n):
        for j in range(n):
            if i == j:
                P_hat[i,j] = 1 - sp.Rational(ri[i], r)
            else:
                P_hat[i,j] = sp.Rational(R[i,j], r)

    # Algoritmo de uniformización
    A = sp.eye(n)
    c = float(sp.exp(-r*t))
    B = c*A
    suma = c
    k = 1

    while (1 - suma) > eps:
        A = A*P_hat
        c = c*(r*t)/k
        B = B + c*A
        suma += c
        k += 1
    return sp.N(B,10), k-1

Repita el ejercicio 3 aplicando este algoritmo con una tolerancia $ε = 0.00001 $ (indique el valor correspondiente de M en cada caso). Compare los resultados.

In [4]:
R = sp.Matrix([
    [0,2,3,0],
    [4,0,2,0],
    [0,2,0,2],
    [1,0,3,0]
])

P05, M05 = uniformizacion(R, 0.5)
P1, M1 = uniformizacion(R, 1)
P5, M5 = uniformizacion(R, 5)

print("M para t=0.5 =", M05)
print("M para t=1 =", M1)
print("M para t=5 =", M5)

print("\nP(0.5)")
sp.pprint(P05)

print("\nP(1)")
sp.pprint(P1)

print("\nP(5)")
sp.pprint(P5)

M para t=0.5 = 13
M para t=1 = 19
M para t=5 = 56

P(0.5)
⎡0.2506079993  0.2169639178  0.3866555748  0.1457691062⎤
⎢                                                      ⎥
⎢0.2531341645  0.2383603034  0.374407879   0.1340942513⎥
⎢                                                      ⎥
⎢0.1691188161  0.1936142079  0.4202996563  0.2169639178⎥
⎢                                                      ⎥
⎣0.1580168021  0.1574439612  0.3983304298  0.286205405 ⎦

P(1)
⎡0.2061503751  0.2039012817  0.3987081057  0.1912350573⎤
⎢                                                      ⎥
⎢0.2082834695  0.205339954   0.3978976845  0.1884737118⎥
⎢                                                      ⎥
⎢0.1967577484  0.1983785906  0.4009571991  0.2039012817⎥
⎢                                                      ⎥
⎣0.1920454785  0.1939964028  0.4014694512  0.2124834873⎦

P(5)
⎡0.1999985263  0.1999985256  0.399997048   0.1999985211⎤
⎢                                                      ⎥
⎢0.199998527   0.1